# 🔤 Tries (Prefix Trees) — Runnable Notebook

Companion to [`README.md`](README.md) and
[`13_tries.html`](13_tries.html).

A tree keyed by characters: insert / search / prefix all cost `O(word length)`, independent of how many words.

## 1. The trie
Each node has a `children` map (char → node) and an `is_word` flag.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}          # char -> TrieNode
        self.is_word = False        # does a word END here?

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for ch in word:                          # walk/create one node per character
            node = node.children.setdefault(ch, TrieNode())
        node.is_word = True                       # mark the last node as a word end

    def _walk(self, s):
        node = self.root
        for ch in s:
            if ch not in node.children:
                return None                       # fell off the trie
            node = node.children[ch]
        return node

    def search(self, word):
        node = self._walk(word)
        return node is not None and node.is_word  # must land AND be a word

    def starts_with(self, prefix):
        return self._walk(prefix) is not None     # just needs to land

t = Trie()
for w in ["cat", "car", "card", "dog"]:
    t.insert(w)
print("search 'car'    :", t.search("car"))
print("search 'ca'     :", t.search("ca"))       # a prefix, not a stored word
print("starts_with 'ca':", t.starts_with("ca"))
print("starts_with 'do':", t.starts_with("do"))
assert t.search("car") and not t.search("ca")
assert t.starts_with("ca") and not t.starts_with("zz")

## 2. Autocomplete — walk to the prefix, DFS-collect the subtree
This is the thing a hash set cannot do without scanning everything.

In [ ]:
def words_with_prefix(trie, prefix):
    node = trie._walk(prefix)
    out = []
    def dfs(n, path):
        if n.is_word:
            out.append(prefix + path)
        for ch, child in sorted(n.children.items()):   # sorted -> alphabetical order
            dfs(child, path + ch)
    if node:
        dfs(node, "")
    return out

print("autocomplete 'ca':", words_with_prefix(t, "ca"))
print("autocomplete 'd' :", words_with_prefix(t, "d"))
assert words_with_prefix(t, "ca") == ["car", "card", "cat"]
assert words_with_prefix(t, "z") == []

## 3. Prefixes are shared (that's the whole point)
`car` and `card` reuse the same `c-a-r` path — only `card` adds one node.

In [ ]:
# From the root, 'c' has one child 'a'; 'a' has children 't' and 'r'; 'r' has child 'd'.
node_c = t.root.children["c"]
node_ca = node_c.children["a"]
print("children after 'ca':", sorted(node_ca.children))   # ['r', 't']
print("'car' is a word?    :", node_ca.children["r"].is_word)
print("'card' continues from 'car':", sorted(node_ca.children["r"].children))  # ['d']
assert sorted(node_ca.children) == ["r", "t"]              # cat and car share 'ca'
assert node_ca.children["r"].is_word                       # 'car' ends here
assert "d" in node_ca.children["r"].children               # 'card' continues

## ✅ Recap
- A trie stores strings **character by character**; shared prefixes share nodes.
- Node = `children` map + **`is_word`** flag (a node existing ≠ a word ending there).
- insert / search / `starts_with` are all **`O(L)`**, independent of word count.
- Beats a hash set for **prefix / autocomplete** (walk to the node, collect the subtree).

Next: [`14_AStar_Floyd_Warshall`](../14_AStar_Floyd_Warshall/README.md).